# Video Generation

From images to video: the core addition is **temporal modeling**. Each frame is generated not independently but conditioned on other frames.

Key approaches:
1. **Factored spatial+temporal attention** (SVD, AnimateDiff) -- spatial attention within each frame, temporal attention across frames
2. **3D attention / spacetime patches** (Sora) -- treat video as 3D volume, full attention over all spacetime tokens
3. **Autoregressive frame prediction** (Gen-4.5 A2D architecture) -- generate frames sequentially, each conditioned on previous

**Time:** ~2-3 hrs | **Hardware:** ~6-8 GB VRAM

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from PIL import Image
from pathlib import Path
import gc
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM: {vram_gb:.1f} GB")
else:
    vram_gb = 0
    print("WARNING: No GPU detected. Video generation will be very slow on CPU.")

output_dir = Path("./outputs")
output_dir.mkdir(exist_ok=True)

In [ ]:
# EXERCISE: Implement temporal self-attention for video

class TemporalSelfAttention(nn.Module):
    """Self-attention across the temporal (frame) dimension.

    Rearranges (B, C, T, H, W) so attention operates across T
    for each spatial position, then rearranges back.
    """

    def __init__(self, dim: int, n_heads: int = 4):
        super().__init__()
        # YOUR CODE: define Q/K/V projections and output projection
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, C, T, H, W). Returns: (B, C, T, H, W)."""
        # YOUR CODE
        raise NotImplementedError


# Test
ta = TemporalSelfAttention(dim=64, n_heads=4)
test_x = torch.randn(2, 64, 8, 16, 16)  # B=2, C=64, T=8, 16x16
out = ta(test_x)
assert out.shape == test_x.shape
print(f"Temporal attention: {test_x.shape} -> {out.shape}")


In [ ]:
# GPU: ~6-8 GB VRAM for Wan2.1-1.3B; fallback to AnimateDiff (~4 GB)
# Load a video generation model -- try smallest first

model_loaded = None

# Strategy: try models from smallest to largest VRAM requirement
# 1. Wan2.1-T2V-1.3B (~6-8 GB) -- newest, smallest text-to-video
# 2. AnimateDiff + SD 1.5 (~4-6 GB) -- factored temporal attention
# 3. SVD (image-to-video) (~6 GB) -- if we have a source image

# --- Try Wan2.1-T2V-1.3B first ---
try:
    from diffusers import WanPipeline
    print("Attempting to load Wan2.1-T2V-1.3B...")
    pipe = WanPipeline.from_pretrained(
        "Wan-AI/Wan2.1-T2V-1.3B",
        torch_dtype=dtype,
    )
    pipe.enable_model_cpu_offload()
    model_loaded = "wan2.1"
    print("Wan2.1-T2V-1.3B loaded successfully.")
except Exception as e:
    print(f"Could not load Wan2.1: {e}")

# --- Fallback to AnimateDiff ---
if model_loaded is None:
    try:
        from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler
        print("\nAttempting to load AnimateDiff...")
        adapter = MotionAdapter.from_pretrained(
            "guoyww/animatediff-motion-adapter-v1-5-3",
            torch_dtype=dtype,
        )
        pipe = AnimateDiffPipeline.from_pretrained(
            "stable-diffusion-v1-5/stable-diffusion-v1-5",
            motion_adapter=adapter,
            torch_dtype=dtype,
        )
        pipe.scheduler = DDIMScheduler.from_config(
            pipe.scheduler.config,
            beta_schedule="linear",
            clip_sample=False,
        )
        pipe.enable_model_cpu_offload()
        model_loaded = "animatediff"
        print("AnimateDiff loaded successfully.")
    except Exception as e:
        print(f"Could not load AnimateDiff: {e}")

# --- Last resort: SVD (image-to-video) ---
if model_loaded is None:
    try:
        from diffusers import StableVideoDiffusionPipeline
        print("\nAttempting to load SVD (image-to-video)...")
        pipe = StableVideoDiffusionPipeline.from_pretrained(
            "stabilityai/stable-video-diffusion-img2vid",
            torch_dtype=dtype,
            variant="fp16",
        )
        pipe.enable_model_cpu_offload()
        model_loaded = "svd"
        print("SVD loaded successfully.")
    except Exception as e:
        print(f"Could not load SVD: {e}")

if model_loaded is None:
    print("\nERROR: No video model could be loaded.")
    print("Install requirements: pip install diffusers transformers accelerate")
else:
    print(f"\nActive model: {model_loaded}")

In [ ]:
# GPU: ~6-8 GB VRAM
# Generate video from text prompt

from diffusers.utils import export_to_video

prompt = "a cat walking across a sunlit room"

print(f"Model: {model_loaded}")
print(f"Prompt: '{prompt}'")
print("Generating video...")

torch.manual_seed(42)
t_start = time.time()

if model_loaded == "wan2.1":
    output = pipe(
        prompt=prompt,
        num_frames=16,
        num_inference_steps=20,
        guidance_scale=5.0,
        height=480,
        width=832,
    )
    frames = output.frames[0]  # list of PIL images

elif model_loaded == "animatediff":
    output = pipe(
        prompt=prompt,
        num_frames=16,
        num_inference_steps=25,
        guidance_scale=7.5,
    )
    frames = output.frames[0]

elif model_loaded == "svd":
    # SVD needs an input image -- generate one with SD
    from diffusers import StableDiffusionPipeline
    sd_pipe = StableDiffusionPipeline.from_pretrained(
        "stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=dtype
    )
    sd_pipe.enable_model_cpu_offload()
    input_image = sd_pipe(prompt, num_inference_steps=25).images[0]
    input_image = input_image.resize((1024, 576))
    del sd_pipe
    gc.collect()
    torch.cuda.empty_cache() if device == "cuda" else None

    output = pipe(input_image, num_frames=14, num_inference_steps=25, decode_chunk_size=4)
    frames = output.frames[0]
else:
    raise RuntimeError("No video model loaded. Run the previous cell.")

gen_time = time.time() - t_start
print(f"Generated {len(frames)} frames in {gen_time:.1f}s")
print(f"Frame size: {frames[0].size if hasattr(frames[0], 'size') else 'N/A'}")

# Export video
video_path = str(output_dir / "generated_video.mp4")
export_to_video(frames, video_path, fps=8)
print(f"Video saved to: {video_path}")

# Display frame grid (every Nth frame)
n_frames = len(frames)
step = max(1, n_frames // 8)
display_frames = frames[::step]

fig, axes = plt.subplots(1, len(display_frames), figsize=(4 * len(display_frames), 4))
if len(display_frames) == 1:
    axes = [axes]

for idx, (ax, frame) in enumerate(zip(axes, display_frames)):
    if isinstance(frame, Image.Image):
        ax.imshow(frame)
    else:
        ax.imshow(frame)
    ax.set_title(f"Frame {idx * step}", fontsize=10)
    ax.axis("off")

fig.suptitle(f'"{prompt}" -- {n_frames} frames @ 8fps ({gen_time:.1f}s)', fontsize=12)
plt.tight_layout()
plt.show()

# Try inline video display (works in Jupyter)
try:
    from IPython.display import Video, display
    display(Video(video_path, embed=True, width=600))
except Exception:
    print(f"Inline video display not available. View at: {video_path}")

In [ ]:
# Inspect model architecture: find temporal attention layers
# This reveals how the model handles time -- the key architectural question

temporal_keywords = ["temporal", "time", "motion", "frame"]

# Get the main model (transformer or unet depending on pipeline)
if hasattr(pipe, 'transformer'):
    main_model = pipe.transformer
    model_type = "Transformer"
elif hasattr(pipe, 'unet'):
    main_model = pipe.unet
    model_type = "U-Net"
else:
    main_model = None
    model_type = "Unknown"

if main_model is not None:
    total_params = sum(p.numel() for p in main_model.parameters()) / 1e6
    print(f"Main model: {model_type} ({total_params:.1f}M params)")
    print(f"\n{'=' * 80}")
    print(f"TEMPORAL / TIME-RELATED LAYERS")
    print(f"{'=' * 80}")

    temporal_layers = []
    spatial_layers = []
    cross_attn_layers = []
    other_layers = []

    for name, module in main_model.named_modules():
        name_lower = name.lower()
        module_type = type(module).__name__

        # Classify layers
        if any(k in name_lower for k in temporal_keywords):
            temporal_layers.append((name, module_type))
        elif "cross" in name_lower and "attn" in name_lower:
            cross_attn_layers.append((name, module_type))
        elif "attn" in name_lower and "self" in name_lower:
            spatial_layers.append((name, module_type))

    print(f"\nTemporal layers ({len(temporal_layers)} found):")
    for name, mtype in temporal_layers[:20]:  # Show first 20
        print(f"  {name}: {mtype}")
    if len(temporal_layers) > 20:
        print(f"  ... and {len(temporal_layers) - 20} more")

    print(f"\nSpatial self-attention layers: {len(spatial_layers)}")
    print(f"Cross-attention layers: {len(cross_attn_layers)}")
    print(f"Temporal layers: {len(temporal_layers)}")

    # Top-level architecture tree
    print(f"\n{'=' * 80}")
    print(f"TOP-LEVEL ARCHITECTURE")
    print(f"{'=' * 80}")
    for name, child in main_model.named_children():
        child_params = sum(p.numel() for p in child.parameters()) / 1e6
        if child_params > 0.01:
            print(f"  {name}: {type(child).__name__} ({child_params:.1f}M)")
else:
    print("Could not find main model in pipeline.")

In [ ]:
# Visualize temporal attention patterns
# Hook into a temporal attention layer and capture attention weights during generation

attention_maps = []
hook_handles = []

def find_temporal_attention(model, keywords=None):
    """Find the first temporal attention module in the model."""
    if keywords is None:
        keywords = ["temporal_attn", "temp_attn", "temporal", "time_mixer", "motion"]

    candidates = []
    for name, module in model.named_modules():
        name_lower = name.lower()
        module_type = type(module).__name__.lower()
        if any(k in name_lower or k in module_type for k in keywords):
            if hasattr(module, 'forward') and 'attention' in module_type.lower() or 'attn' in name_lower:
                candidates.append((name, module))
    return candidates


def attention_hook(module, input, output):
    """Capture attention weights from a temporal attention layer."""
    # Different models store attention differently
    # Try to extract attention weights from the output
    if isinstance(output, tuple) and len(output) > 1:
        # Some models return (hidden_states, attention_weights)
        attn_weights = output[1]
        if attn_weights is not None:
            attention_maps.append(attn_weights.detach().cpu().float())


# Try to hook temporal attention
if main_model is not None:
    candidates = find_temporal_attention(main_model)
    if candidates:
        print(f"Found {len(candidates)} temporal attention candidates.")
        # Hook the first few
        for name, module in candidates[:3]:
            handle = module.register_forward_hook(attention_hook)
            hook_handles.append(handle)
            print(f"  Hooked: {name}")

        # Generate a short video to capture attention
        print("\nGenerating short video to capture attention patterns...")
        torch.manual_seed(123)
        attention_maps.clear()

        if model_loaded == "wan2.1":
            _ = pipe(
                prompt="a ball bouncing",
                num_frames=8,
                num_inference_steps=5,  # Few steps, just to capture attention
                guidance_scale=5.0,
                height=480,
                width=832,
            )
        elif model_loaded == "animatediff":
            _ = pipe(
                prompt="a ball bouncing",
                num_frames=8,
                num_inference_steps=5,
                guidance_scale=7.5,
            )

        # Remove hooks
        for h in hook_handles:
            h.remove()

        print(f"Captured {len(attention_maps)} attention maps.")
    else:
        print("No temporal attention layers found with standard names.")
        print("This model may use a different temporal mechanism.")

# Visualize captured attention maps OR create synthetic illustration
if attention_maps:
    fig, axes = plt.subplots(1, min(3, len(attention_maps)), figsize=(6 * min(3, len(attention_maps)), 5))
    if not isinstance(axes, np.ndarray):
        axes = [axes]

    for idx, (ax, attn) in enumerate(zip(axes, attention_maps[:3])):
        # Average over batch and heads to get frame-to-frame attention
        if attn.dim() == 4:  # (batch, heads, seq, seq)
            attn_avg = attn[0].mean(dim=0).numpy()  # Average over heads
        elif attn.dim() == 3:  # (batch, seq, seq)
            attn_avg = attn[0].numpy()
        else:
            attn_avg = attn.numpy()

        # If the attention map is larger than num_frames, it includes spatial tokens
        # Try to reshape to show frame-level patterns
        im = ax.imshow(attn_avg[:min(32, attn_avg.shape[0]), :min(32, attn_avg.shape[1])],
                       cmap='viridis', aspect='auto')
        ax.set_title(f"Temporal Attn Map {idx}\n({attn_avg.shape})", fontsize=11)
        ax.set_xlabel("Key position")
        ax.set_ylabel("Query position")
        plt.colorbar(im, ax=ax, fraction=0.046)

    plt.suptitle("Captured Temporal Attention Patterns", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    # Create a synthetic illustration of temporal attention
    print("Creating synthetic temporal attention illustration...")
    n_frames = 16

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1. Factored temporal attention (SVD/AnimateDiff style)
    # Each frame attends to all other frames at the same spatial position
    factored_attn = np.zeros((n_frames, n_frames))
    for i in range(n_frames):
        for j in range(n_frames):
            # Gaussian decay with distance, strong self-attention
            factored_attn[i, j] = np.exp(-0.3 * abs(i - j))
    factored_attn /= factored_attn.sum(axis=1, keepdims=True)

    im0 = axes[0].imshow(factored_attn, cmap='viridis', aspect='auto')
    axes[0].set_title("Factored Temporal\n(SVD/AnimateDiff)", fontsize=12)
    axes[0].set_xlabel("Key frame")
    axes[0].set_ylabel("Query frame")
    plt.colorbar(im0, ax=axes[0], fraction=0.046)

    # 2. Full 3D attention (Sora-like)
    # Every spacetime token attends to every other -- show frame-averaged pattern
    full_3d = np.ones((n_frames, n_frames)) / n_frames  # Roughly uniform
    # Add slight temporal locality bias
    for i in range(n_frames):
        for j in range(n_frames):
            full_3d[i, j] += 0.05 * np.exp(-0.1 * abs(i - j))
    full_3d /= full_3d.sum(axis=1, keepdims=True)

    im1 = axes[1].imshow(full_3d, cmap='viridis', aspect='auto')
    axes[1].set_title("Full 3D Spacetime\n(Sora-like)", fontsize=12)
    axes[1].set_xlabel("Key frame")
    axes[1].set_ylabel("Query frame")
    plt.colorbar(im1, ax=axes[1], fraction=0.046)

    # 3. Causal/autoregressive (A2D-style)
    # Each frame only attends to previous frames
    causal_attn = np.zeros((n_frames, n_frames))
    for i in range(n_frames):
        for j in range(i + 1):  # Only past + current
            causal_attn[i, j] = np.exp(-0.2 * (i - j))
    causal_attn /= causal_attn.sum(axis=1, keepdims=True)

    im2 = axes[2].imshow(causal_attn, cmap='viridis', aspect='auto')
    axes[2].set_title("Causal/Autoregressive\n(A2D-style)", fontsize=12)
    axes[2].set_xlabel("Key frame")
    axes[2].set_ylabel("Query frame")
    plt.colorbar(im2, ax=axes[2], fraction=0.046)

    plt.suptitle("Temporal Attention Patterns (Synthetic Illustration)", fontsize=14)
    plt.tight_layout()
    plt.show()

    print("\nFactored: Each frame attends to all frames at same spatial position. Banded pattern = temporal locality.")
    print("Full 3D: All spacetime tokens attend to each other. Near-uniform when averaged over space.")
    print("Causal: Lower-triangular -- each frame only sees past frames. Recent frames get more attention.")

In [ ]:
# GPU: ~6-8 GB VRAM per generation
# Experiment: vary number of frames and observe temporal consistency + timing

frame_counts = [8, 16, 24]  # Adjust based on what the model supports
test_prompt = "ocean waves crashing on a rocky shore"
results = {}

print(f"Prompt: '{test_prompt}'")
print(f"Testing frame counts: {frame_counts}\n")

for n_frames in frame_counts:
    try:
        torch.manual_seed(42)  # Same seed for fair comparison
        t_start = time.time()

        if model_loaded == "wan2.1":
            output = pipe(
                prompt=test_prompt,
                num_frames=n_frames,
                num_inference_steps=15,
                guidance_scale=5.0,
                height=480,
                width=832,
            )
        elif model_loaded == "animatediff":
            output = pipe(
                prompt=test_prompt,
                num_frames=n_frames,
                num_inference_steps=20,
                guidance_scale=7.5,
            )
        elif model_loaded == "svd":
            # SVD has a fixed frame count, skip this experiment
            print("SVD has fixed frame count -- skipping frame count experiment.")
            break

        gen_time = time.time() - t_start
        result_frames = output.frames[0]
        results[n_frames] = {
            "frames": result_frames,
            "time": gen_time,
            "time_per_frame": gen_time / n_frames,
        }
        print(f"  {n_frames} frames: {gen_time:.1f}s total, {gen_time/n_frames:.2f}s/frame")

        # Save video
        video_path = str(output_dir / f"video_{n_frames}frames.mp4")
        export_to_video(result_frames, video_path, fps=8)

    except Exception as e:
        print(f"  {n_frames} frames: FAILED -- {e}")
        # Try to free memory
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

# Visualize: first, middle, last frame for each frame count
if results:
    n_results = len(results)
    fig, axes = plt.subplots(n_results, 3, figsize=(12, 4 * n_results))
    if n_results == 1:
        axes = axes[np.newaxis, :]

    for row, (n_frames, res) in enumerate(results.items()):
        frames_list = res["frames"]
        mid_idx = len(frames_list) // 2

        for col, (frame_idx, label) in enumerate([
            (0, "First frame"),
            (mid_idx, "Middle frame"),
            (len(frames_list) - 1, "Last frame"),
        ]):
            axes[row, col].imshow(frames_list[frame_idx])
            axes[row, col].set_title(f"{label} (#{frame_idx})", fontsize=10)
            axes[row, col].axis("off")

        axes[row, 0].set_ylabel(
            f"{n_frames} frames\n{res['time']:.1f}s",
            fontsize=12, fontweight='bold', rotation=0, labelpad=80
        )

    fig.suptitle(f"Frame Count Comparison: '{test_prompt}'", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

    # Timing chart
    fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    frame_nums = list(results.keys())
    total_times = [results[n]["time"] for n in frame_nums]
    per_frame = [results[n]["time_per_frame"] for n in frame_nums]

    ax1.bar(range(len(frame_nums)), total_times, color='#3498db', edgecolor='black')
    ax1.set_xticks(range(len(frame_nums)))
    ax1.set_xticklabels([f"{n} frames" for n in frame_nums])
    ax1.set_ylabel("Total time (s)")
    ax1.set_title("Total Generation Time")
    for i, t in enumerate(total_times):
        ax1.text(i, t + 0.5, f"{t:.1f}s", ha='center', fontweight='bold')

    ax2.bar(range(len(frame_nums)), per_frame, color='#e74c3c', edgecolor='black')
    ax2.set_xticks(range(len(frame_nums)))
    ax2.set_xticklabels([f"{n} frames" for n in frame_nums])
    ax2.set_ylabel("Time per frame (s)")
    ax2.set_title("Per-Frame Cost")
    for i, t in enumerate(per_frame):
        ax2.text(i, t + 0.02, f"{t:.2f}s", ha='center', fontweight='bold')

    plt.tight_layout()
    plt.show()

    print("\nNote: Per-frame cost may increase with more frames due to longer temporal attention sequences.")
else:
    print("No results to display.")

## Temporal Modeling Approaches

### 1. Factored Attention (SVD, AnimateDiff)
```
For each denoising step:
  For each layer:
    Spatial attention: tokens within frame i attend to each other
    Temporal attention: token at position (x,y) attends across all frames
    Cross attention: all tokens attend to text embeddings
```
- **Pro:** Memory efficient -- spatial is O(HW)^2, temporal is O(T)^2, not O(THW)^2
- **Con:** Limited spatial-temporal interaction -- a moving object can't attend to its future position in another frame

### 2. 3D / Spacetime Patches (Sora-like)
```
Video (T x H x W) → patchify in 3D → (T/pt x H/ph x W/pw) tokens
Full self-attention over ALL spacetime tokens
```
- **Pro:** Rich interactions -- any spacetime location can attend to any other
- **Con:** O(N^2) over all spacetime tokens. For 16 frames at 32x32 latent with 2x2x1 patches = 4,096 tokens. Quadratic cost makes scaling to longer videos very expensive.

### 3. Autoregressive (Gen-4.5 A2D, GWM-1)
```
Generate frame 0 (or chunk of frames)
For each subsequent chunk:
  Condition on previous frames → generate next frames
  Causal mask: can only attend to past
```
- **Pro:** Can stream output in real-time, naturally handles variable-length video, constant memory per chunk
- **Con:** Sequential bottleneck (can't parallelize across time), potential error accumulation over long sequences

**Why AR has been chosen by several leading labs:** It enables real-time interactive generation (users see frames as they're produced), naturally scales to arbitrarily long videos without quadratic cost, and aligns with the LLM-style scaling paradigm that has proven most effective.

## Checkpoint

**3 approaches to temporal modeling:**

| Approach | Models | Temporal Mechanism | Scaling | Streaming |
|----------|--------|-------------------|---------|----------|
| Factored spatial+temporal | SVD, AnimateDiff | Separate spatial and temporal attention | O(T^2 + (HW)^2) | No |
| Full 3D attention | Sora-style | All spacetime tokens attend to each other | O((THW)^2) | No |
| Autoregressive | Gen-4.5 A2D, GWM-1 | Causal -- each chunk conditioned on previous | O(T) chunks, each O(chunk_size) | Yes |

**Why autoregressive is a prominent direction in the field:**
1. Enables real-time interaction (users see frames as they're produced)
2. Naturally scales to long videos (no quadratic blowup over time)
3. Aligns with the LLM scaling paradigm that has proven most effective
4. CausVid (2412.07772) showed that causal generation can match bidirectional quality with distillation

**Further reading:**
- SVD: [2311.15127](https://arxiv.org/abs/2311.15127)
- Sora technical report (OpenAI blog)
- Wan2.1: [2503.20314](https://arxiv.org/abs/2503.20314)
- CausVid: [2412.07772](https://arxiv.org/abs/2412.07772)